## Step 1: Setting up our environment

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch import nn,optim

## Step 2: Defining the GAN Framework

In [9]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator,self).__init__()
        self.hidden_layer=nn.Linear(100,256)
        self.output_layer=nn.Linear(256,784)

    def forward(self,x):
        x=torch.relu(self.hidden_layer(x))
        x=torch.tanh(self.output_layer(x))
        return x

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator,self).__init__()
        self.hidden_layer=nn.Linear(784,256)
        self.output_layer=nn.Linear(256,1)

    def forward(self,x):
        x=torch.relu(self.hidden_layer(x))
        x=torch.sigmoid(self.output_layer(x))
        return x

## Step 3: Training the GAN

In [11]:
generator = Generator()
discriminator = Discriminator()

#Optimizers and loss function
optimizer_g=optim.Adam(generator.parameters(), lr =0.0002)
optimizer_d=optim.Adam(discriminator.parameters(), lr =0.0002)
criterion=nn.BCELoss()
batch_size=32

for epoch in range (100):
    #Generate fake data
    noise=torch.randn(batch_size,100)
    fake_data=generator(noise)

    #Train Discriminator
    #Here the assumption is that the real data comes from the uniform distribution between -1 and 1
    real_data = torch.randn(batch_size,784)*2-1 #Load real data here
    optimizer_d.zero_grad()
    real_loss=criterion(discriminator(real_data),torch.ones(batch_size,1))
    fake_loss=criterion(discriminator(fake_data.detach()),torch.zeros(batch_size,1))
    d_loss=real_loss+fake_loss
    d_loss.backward()
    optimizer_d.step()

    #Train Generator
    optimizer_g.zero_grad()
    g_loss=criterion(discriminator(fake_data),torch.ones(batch_size,1))
    g_loss.backward()
    optimizer_g.step()

    if epoch%10 == 0:
        print(f'Epoch {epoch}, Discriminator Loss:{d_loss.item()}, Generator Loss:{g_loss.item()}')

Epoch 0, Discriminator Loss:1.2702946662902832, Generator Loss:0.7172368764877319
Epoch 10, Discriminator Loss:0.8905264735221863, Generator Loss:0.5524419546127319
Epoch 20, Discriminator Loss:0.973508894443512, Generator Loss:0.5109511613845825
Epoch 30, Discriminator Loss:0.8206532597541809, Generator Loss:0.6513034701347351
Epoch 40, Discriminator Loss:0.7029777765274048, Generator Loss:0.8735911846160889
Epoch 50, Discriminator Loss:0.9497100114822388, Generator Loss:0.6758037805557251
Epoch 60, Discriminator Loss:1.1135411262512207, Generator Loss:0.5876567363739014
Epoch 70, Discriminator Loss:0.9171614050865173, Generator Loss:0.6979344487190247
Epoch 80, Discriminator Loss:0.7022351622581482, Generator Loss:0.8901422023773193
Epoch 90, Discriminator Loss:0.7161012291908264, Generator Loss:0.8553987145423889


## Step 4: Creating the VAE Framework

In [17]:
class VAE(nn.Module):
    def __init__(self):
        super(VAE,self).__init__()
        self.encoder_hidden=nn.Linear(784,256)
        self.mean_layer=nn.Linear(256,20)
        self.log_var_layer=nn.Linear(256,20)
        self.decoder_hidden=nn.Linear(20,256)
        self.output_layer=nn.Linear(256,784)

    def encode(self,x):
        h=torch.relu(self.encoder_hidden(x))
        return self.mean_layer(h),self.log_var_layer(h)

    def decode(self,z):
        h=torch.relu(self.decoder_hidden(z))
        return torch.sigmoid(self.output_layer(h))

    def reparameterize(self,mu,log_var):
        std=torch.exp(0.5*log_var)
        eps=torch.randn_like(std)
        return mu+eps*std

    def forward(self,x):
        mu,log_var=self.encode(x)
        z=self.reparameterize(mu,log_var)
        return self.decode(z),mu,log_var

## Step 5: Training the VAE

In [19]:
def loss_function(recon_x,x,mu,log_var):
    BCE=nn.functional.binary_cross_entropy(recon_x,x,reduction='sum')
    KLD= -0.5*torch.sum(1+log_var-mu.pow(2)-log_var.exp())
    return BCE+KLD

vae=VAE()
optimizer=optim.Adam(vae.parameters(),lr=0.001)
for epoch in range(100):
    vae.train()
    x=torch.rand(batch_size,784) #load batch of data here
    optimizer.zero_grad()
    recon_x,mu,log_var=vae(x)
    loss=loss_function(recon_x,x,mu,log_var)
    loss.backward()
    optimizer.step()

    if epoch%10 == 0:
        print(f'Epoch {epoch}, Loss:{loss.item()}')

Epoch 0, Loss:17593.513671875
Epoch 10, Loss:17503.650390625
Epoch 20, Loss:17472.318359375
Epoch 30, Loss:17462.46875
Epoch 40, Loss:17470.208984375
Epoch 50, Loss:17455.5234375
Epoch 60, Loss:17438.673828125
Epoch 70, Loss:17442.595703125
Epoch 80, Loss:17439.818359375
Epoch 90, Loss:17442.71484375


## Step 6: Analyzing Sequence Prediction

In [22]:
#Assume existing sequence 
sequence_length=5
batch_size=10
input_size=10
hidden_size=20
num_layers=2
vocab_size=10

sequence_data=torch.randn(batch_size,sequence_length,input_size) #(B,Seq,Input)
model=nn.LSTM(input_size=input_size,hidden_size=hidden_size,num_layers=num_layers)

#Target needs to be *class indices* (long integers)
sequence_data_target=torch.randint(0,vocab_size,(batch_size,sequence_length)).long() #(B,Seq)

def train_autoregressive(sequence_data):
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.Adam(model.parameters(),lr=0.01)
    for epoch in range(50):
        model.train()
        optimizer.zero_grad()
        output,_=model(sequence_data) #LSTM Outputs (batch,seq_length,hidden_size)

        #Reshape: (B,Seq,Hidden) -> (B*Seq,Hidden) to match Cross Entropy Loss
        loss=criterion(output.reshape(-1,hidden_size), sequence_data_target.reshape(-1))
        loss.backward()
        optimizer.step()

        if epoch%10 == 0:
            print(f'Epoch {epoch}, Loss:{loss.item()}')

train_autoregressive(sequence_data)
        

Epoch 0, Loss:2.9526543617248535
Epoch 10, Loss:2.515474319458008
Epoch 20, Loss:2.366520881652832
Epoch 30, Loss:2.3017821311950684
Epoch 40, Loss:2.231602668762207
